# 04 — ML Signal Strategy Analysis

This notebook analyses the walk-forward ML classification strategy implemented
in `strategies/ml_signal.py`.

**Sections**
- Feature engineering inspection
- Target variable inspection
- Walk-forward validation structure
- Baseline run & equity curve
- Feature importance analysis
- Signal analysis
- Model type comparison (XGBoost vs Random Forest vs Logistic)
- Three-strategy comparison: ML vs Momentum vs Mean Reversion
- Automated key findings

**Background — Walk-forward ML signal**:
A machine learning model predicts whether each stock will outperform the
cross-sectional median return over the next N days.  The model is trained on
an *expanding window* of past data and generates predictions only for the
out-of-sample future — the only valid way to use supervised ML in a backtest.

**Why in-sample fitting is lookahead bias**: If you fit a model on the full
sample (2015–2024) and evaluate it on the same period, the model has already
seen the future outcomes it is predicting. This is as severe as using
tomorrow's prices today — the model's apparent performance reflects memorization,
not genuine predictive skill. The walk-forward design prevents this by ensuring
that predictions at time T are made by a model that has never seen any data
after T.

In [ ]:
%matplotlib inline
import sys
import warnings
import pathlib

_here = pathlib.Path(".").resolve()
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from backtester import DataLoader, Backtester, compute_metrics, drawdown_series
from strategies.ml_signal import MLSignalStrategy, compute_features, compute_target
from strategies.momentum import MomentumStrategy
from strategies.mean_reversion import MeanReversionStrategy
from config import DEFAULT_TICKERS, BENCHMARK_TICKER, START_DATE, END_DATE, INITIAL_CAPITAL

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 5)

print("Environment ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Load price data — `prices` is the canonical wide-format DataFrame used
# throughout this notebook (index=DatetimeIndex, columns=ticker symbols).
# ---------------------------------------------------------------------------
all_tickers = list(DEFAULT_TICKERS) + [BENCHMARK_TICKER]

loader = DataLoader(tickers=all_tickers, start_date=START_DATE, end_date=END_DATE)
prices = loader.load_wide()

n_tickers = len(prices.columns)
n_days    = len(prices)
print(f"prices: {n_tickers} tickers x {n_days} trading days")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail(3)

In [ ]:
# ---------------------------------------------------------------------------
# Feature engineering inspection.
# ---------------------------------------------------------------------------
features = compute_features(prices)

print(f"Feature DataFrame shape: {features.shape}")
print(f"Feature columns ({len(features.columns)}): {features.columns.tolist()}")
print(f"\nMissing value counts per feature:")
print(features.isna().sum())
print(f"\nDescriptive statistics:")
features.describe().T

In [ ]:
# ---------------------------------------------------------------------------
# Target variable inspection.
# ---------------------------------------------------------------------------
target = compute_target(prices, forward_days=5)
target_clean = target.dropna()

print(f"Target shape (long format): {target_clean.shape}")
print(f"Class balance: {target_clean.mean():.4f} (1=outperform, 0=underperform)")
print(f"  Class 1 (outperform): {int(target_clean.sum()):,} obs ({target_clean.mean()*100:.1f}%)")
print(f"  Class 0 (underperform): {int((1-target_clean).sum()):,} obs ({(1-target_clean.mean())*100:.1f}%)")

# Histogram of 5-day forward returns colored by target class
fwd_return = (prices.shift(-5) / prices - 1).stack(future_stack=True)
fwd_return.index.names = ["date", "ticker"]
fwd_df = pd.DataFrame({"fwd_return": fwd_return, "target": target_clean}).dropna()
fwd_df["fwd_return_pct"] = fwd_df["fwd_return"] * 100

fig, ax = plt.subplots(figsize=(12, 5))
for t_val, label, color in [(1, "Outperform (class=1)", "steelblue"),
                             (0, "Underperform (class=0)", "firebrick")]:
    subset = fwd_df[fwd_df["target"] == t_val]["fwd_return_pct"]
    ax.hist(subset, bins=80, alpha=0.5, label=label, color=color, density=True)
ax.axvline(0, color="black", linewidth=1.2, linestyle="--")
ax.set_title("5-day forward returns by target class", fontweight="bold")
ax.set_xlabel("5-day forward return (%)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Section 1 — Walk-forward structure

In [ ]:
# ---------------------------------------------------------------------------
# Walk-forward timeline visualization.
# A Gantt-style chart makes the expanding-window structure visually obvious:
# each training bar grows longer; each prediction bar is a fixed-width slot.
# ---------------------------------------------------------------------------
strategy = MLSignalStrategy(
    model_type="xgboost",
    min_train_years=3,
    retrain_freq_months=3,
)
retraining_dates = strategy._get_retraining_dates(prices.index)

data_start = prices.index[0]
data_end   = prices.index[-1]

fig, ax = plt.subplots(figsize=(16, max(4, len(retraining_dates) * 0.35 + 1)))

for i, train_cutoff in enumerate(retraining_dates):
    predict_start = train_cutoff + pd.Timedelta(days=1)
    predict_end = (
        retraining_dates[i + 1] - pd.Timedelta(days=1)
        if i + 1 < len(retraining_dates)
        else data_end
    )
    # Training bar: data_start → train_cutoff (expanding)
    ax.barh(
        i, (train_cutoff - data_start).days, left=0,
        height=0.5, color="steelblue", alpha=0.7,
        label="Training" if i == 0 else ""
    )
    # Prediction bar: train_cutoff → predict_end
    ax.barh(
        i, (predict_end - train_cutoff).days,
        left=(train_cutoff - data_start).days,
        height=0.5, color="mediumseagreen", alpha=0.8,
        label="Prediction" if i == 0 else ""
    )

# Convert x-axis from days-offset to calendar dates
total_days = (data_end - data_start).days
tick_years = pd.date_range(data_start, data_end, freq="YS")
tick_offsets = [(d - data_start).days for d in tick_years]
ax.set_xticks(tick_offsets)
ax.set_xticklabels([d.strftime("%Y") for d in tick_years], rotation=0)

ax.set_yticks(range(len(retraining_dates)))
ax.set_yticklabels([f"Window {i+1}" for i in range(len(retraining_dates))],
                   fontsize=8)
ax.set_xlabel("Calendar date")
ax.set_title("Walk-forward validation schedule (expanding training window)",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Total windows: {len(retraining_dates)}")
print(f"First training cutoff: {retraining_dates[0].date()}")
print(f"Last training cutoff:  {retraining_dates[-1].date()}")

In [ ]:
# ---------------------------------------------------------------------------
# Single strategy run with default parameters.
# ---------------------------------------------------------------------------
bt = Backtester(
    prices,
    config={
        "initial_capital": INITIAL_CAPITAL,
        "commission_bps":  5,
        "slippage_bps":    2,
        "position_sizing": "equal_weight",
        "benchmark":       BENCHMARK_TICKER,
        "allow_short":     False,
    },
)

result = bt.run(strategy)
m      = result.metrics
print(f"\nBacktest complete — {len(result.trades)} trades executed.")

In [ ]:
# ---------------------------------------------------------------------------
# Metrics summary.
# ---------------------------------------------------------------------------
def _fmt(v):
    if v is None:            return "N/A"
    if isinstance(v, int):   return f"{v:,}"
    if isinstance(v, float): return f"{v:.4f}"
    return str(v)

SEP = "-" * 50
print(f"{'Metric':<40}  {'Value':>10}")
print(SEP)
groups = [
    ("Returns", ["total_return_pct", "annualized_return_pct",
                 "annualized_volatility_pct", "best_day_pct",
                 "worst_day_pct", "positive_days_pct"]),
    ("Risk-adjusted", ["sharpe_ratio", "sortino_ratio", "calmar_ratio"]),
    ("Drawdown", ["max_drawdown_pct", "max_drawdown_duration_days", "recovery_days"]),
    ("Activity", ["n_trades", "win_rate_pct", "avg_trade_duration_days",
                  "total_cost_pct", "turnover_annual_pct"]),
    ("vs Benchmark", ["alpha_pct", "beta", "information_ratio",
                      "correlation_with_benchmark",
                      "benchmark_annualized_return_pct",
                      "benchmark_sharpe_ratio"]),
]
for section, keys in groups:
    print(f"  {section}")
    for k in keys:
        if k in m:
            print(f"    {k:<38}  {_fmt(m[k]):>10}")
    print()

## Section 2 — Equity curve & drawdown

In [ ]:
# ---------------------------------------------------------------------------
# Equity curve vs SPY + drawdown panel.
# ---------------------------------------------------------------------------
bench_result = bt.run_benchmark()
bench_equity = bench_result.equity_curve

strat_norm = result.equity_curve / result.equity_curve.iloc[0] * 100
bench_norm = bench_equity        / bench_equity.iloc[0]        * 100
dd         = drawdown_series(result.equity_curve)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})

axes[0].plot(strat_norm.index, strat_norm.values,
             color="steelblue", linewidth=1.8, label=strategy.get_name())
axes[0].plot(bench_norm.index, bench_norm.values,
             color="black", linewidth=1.4, linestyle="--",
             label=f"{BENCHMARK_TICKER} (buy & hold)")
axes[0].axhline(100, color="gray", linewidth=0.8, linestyle=":", label="Cash")
axes[0].fill_between(
    strat_norm.index, strat_norm.values, bench_norm.values,
    where=(strat_norm.values >= bench_norm.values),
    interpolate=True, alpha=0.15, color="green", label="Outperformance"
)
axes[0].fill_between(
    strat_norm.index, strat_norm.values, bench_norm.values,
    where=(strat_norm.values < bench_norm.values),
    interpolate=True, alpha=0.15, color="red", label="Underperformance"
)
axes[0].set_ylabel("Portfolio value (base = 100)")
axes[0].set_title("Equity curve vs. benchmark", fontweight="bold")
axes[0].legend(fontsize=9)

axes[1].fill_between(dd.index, dd.values, 0,
                     color="firebrick", alpha=0.4, label="Strategy drawdown")
axes[1].plot(dd.index, dd.values, color="firebrick", linewidth=0.8)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown", fontweight="bold")
axes[1].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## Section 3 — Feature importance

In [ ]:
# ---------------------------------------------------------------------------
# Feature importance: mean bar chart + drift heatmap.
# ---------------------------------------------------------------------------
fi = strategy.feature_importances_
mean_fi = fi.mean(axis=0).sort_values(ascending=False)

# Categorize features for colour coding
def feat_color(name: str) -> str:
    if name.startswith("ret_"):         return "steelblue"
    if name.startswith("vol_"):         return "darkorange"
    if name.startswith(("mom_", "rsi_", "bb_")): return "mediumseagreen"
    if name.startswith("price_to_"):    return "firebrick"
    return "gray"

bar_colors = [feat_color(f) for f in mean_fi.index]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: mean importance bar chart
axes[0].bar(range(len(mean_fi)), mean_fi.values, color=bar_colors)
axes[0].set_xticks(range(len(mean_fi)))
axes[0].set_xticklabels(mean_fi.index, rotation=45, ha="right", fontsize=9)
axes[0].set_title("Mean feature importance across all training windows",
                  fontweight="bold")
axes[0].set_ylabel("Mean importance")

# Legend for categories
from matplotlib.patches import Patch
legend_elems = [
    Patch(facecolor="steelblue",      label="Returns (ret_*)"),
    Patch(facecolor="darkorange",     label="Volatility (vol_*)"),
    Patch(facecolor="mediumseagreen", label="Momentum/RSI/BB"),
    Patch(facecolor="firebrick",      label="Price levels"),
]
axes[0].legend(handles=legend_elems, fontsize=8)

# Right: feature importance drift heatmap
fi_ordered = fi[mean_fi.index]  # sort columns by mean importance
x_labels = [d.strftime("%Y-%m") for d in fi_ordered.index]
sns.heatmap(
    fi_ordered.T,
    ax=axes[1],
    cmap="YlOrRd",
    xticklabels=x_labels,
    yticklabels=fi_ordered.columns.tolist(),
    cbar_kws={"label": "Feature importance"},
    linewidths=0.3,
)
axes[1].set_title("Feature importance drift over time", fontweight="bold")
axes[1].set_xlabel("Training window cutoff")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

plt.tight_layout()
plt.show()

## Section 4 — Signal analysis

In [ ]:
# ---------------------------------------------------------------------------
# Distribution of predicted outperformance probabilities.
# Well-calibrated model → roughly uniform distribution.
# ---------------------------------------------------------------------------
proba_wide = strategy.proba_wide_
proba_flat = proba_wide.values.flatten()
proba_flat = proba_flat[~np.isnan(proba_flat)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(proba_flat, bins=50, color="steelblue", edgecolor="white", alpha=0.8, density=True)
ax.axvline(0.5, color="red", linewidth=1.5, linestyle="--", label="P = 0.5")
ax.set_title("Distribution of predicted outperformance probability",
             fontweight="bold")
ax.set_xlabel("P(ticker outperforms median next 5 days)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)
print(f"Probability stats: mean={proba_flat.mean():.4f}, "
      f"std={proba_flat.std():.4f}, "
      f"min={proba_flat.min():.4f}, max={proba_flat.max():.4f}")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Signal heatmap — last 3 years of data.
# ---------------------------------------------------------------------------
raw_signals = strategy.generate_signals(prices)
heatmap_data = raw_signals.iloc[-756:].T  # tickers as rows, dates as columns

step = max(1, len(heatmap_data.columns) // 40)
hm_sub = heatmap_data.iloc[:, ::step]
xlabels = [d.strftime("%Y-%m") for d in hm_sub.columns]

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    hm_sub,
    ax=ax,
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.0,
    xticklabels=xlabels,
    yticklabels=hm_sub.index.tolist(),
    cbar_kws={"label": "Signal (1=long, 0=flat)"},
)
ax.set_title("ML signal — long positions over time (last 3 years)", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Ticker")
ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
plt.show()

## Section 5 — Model comparison

In [ ]:
# ---------------------------------------------------------------------------
# Compare all three model types.
# ---------------------------------------------------------------------------
model_configs = {
    "XGBoost":       MLSignalStrategy(model_type="xgboost",       min_train_years=3, retrain_freq_months=3),
    "Random Forest": MLSignalStrategy(model_type="random_forest",  min_train_years=3, retrain_freq_months=3),
    "Logistic":      MLSignalStrategy(model_type="logistic",        min_train_years=3, retrain_freq_months=3),
}

model_results = {}
for name, strat in model_configs.items():
    r = bt.run(strat)
    model_results[name] = r
    print(f"  [{name:15s}]  Sharpe={r.metrics['sharpe_ratio']:.3f}  "
          f"Ann.Ret={r.metrics['annualized_return_pct']:.1f}%  "
          f"MaxDD={r.metrics['max_drawdown_pct']:.1f}%")

# Equity curve overlay
palette = {"XGBoost": "steelblue", "Random Forest": "mediumseagreen", "Logistic": "darkorange"}
fig, ax = plt.subplots(figsize=(14, 6))

for name, r in model_results.items():
    norm = r.equity_curve / r.equity_curve.iloc[0] * 100
    ax.plot(norm.index, norm.values, color=palette[name], linewidth=1.8, label=name)

bench_norm2 = bench_equity / bench_equity.iloc[0] * 100
ax.plot(bench_norm2.index, bench_norm2.values,
        color="black", linewidth=1.2, linestyle="--", label="SPY")
ax.set_title("Model type comparison: equity curves", fontweight="bold")
ax.set_ylabel("Portfolio value (base = 100)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Metrics comparison table
compare_keys = [
    "annualized_return_pct", "sharpe_ratio", "sortino_ratio",
    "max_drawdown_pct", "calmar_ratio", "alpha_pct",
    "n_trades", "total_cost_pct",
]
bench_m = bench_result.metrics
df_models = pd.DataFrame({
    name: {k: r.metrics.get(k) for k in compare_keys}
    for name, r in model_results.items()
})
df_models["SPY"] = {k: bench_m.get(k) for k in compare_keys}
df_models

## Section 6 — Three-strategy comparison

In [ ]:
# ---------------------------------------------------------------------------
# ML vs Momentum vs Mean Reversion — capstone Phase 3 analysis.
# The pairwise return correlation matrix determines whether combining
# these strategies in Phase 4.1 adds diversification value.
# ---------------------------------------------------------------------------
mom_strategy = MomentumStrategy(lookback_months=12, skip_months=1, n_long=5)
mr_strategy  = MeanReversionStrategy()
ml_strategy  = MLSignalStrategy(model_type="xgboost",
                                 min_train_years=3,
                                 retrain_freq_months=3)

r_mom = bt.run(mom_strategy)
r_mr  = bt.run(mr_strategy)
r_ml  = bt.run(ml_strategy)

# Equity curve overlay (4 curves)
fig, ax = plt.subplots(figsize=(14, 7))
for label, r, col, ls in [
    (ml_strategy.get_name(),  r_ml,  "steelblue",    "-"),
    (mom_strategy.get_name(), r_mom, "mediumpurple", "-"),
    (mr_strategy.get_name(),  r_mr,  "teal",         "-"),
    ("SPY",                   bench_result, "black", "--"),
]:
    norm = r.equity_curve / r.equity_curve.iloc[0] * 100
    ax.plot(norm.index, norm.values, color=col, linewidth=1.8,
            linestyle=ls, label=label)

ax.set_title("ML vs Momentum vs Mean Reversion vs SPY", fontweight="bold")
ax.set_ylabel("Portfolio value (base = 100)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

# Pairwise return correlation matrix
ret_ml  = r_ml.equity_curve.pct_change().dropna()
ret_mom = r_mom.equity_curve.pct_change().dropna()
ret_mr  = r_mr.equity_curve.pct_change().dropna()

common_idx = ret_ml.index.intersection(ret_mom.index).intersection(ret_mr.index)
corr_df = pd.DataFrame({
    "ML":            ret_ml.reindex(common_idx),
    "Momentum":      ret_mom.reindex(common_idx),
    "Mean Reversion": ret_mr.reindex(common_idx),
}).corr()

print("\nPairwise daily return correlation matrix:")
print(corr_df.round(4).to_string())

# Full metrics table
df_all = pd.DataFrame({
    "ML":            {k: r_ml.metrics.get(k)   for k in compare_keys},
    "Momentum":      {k: r_mom.metrics.get(k)  for k in compare_keys},
    "Mean Reversion":{k: r_mr.metrics.get(k)   for k in compare_keys},
    "SPY":           {k: bench_m.get(k)        for k in compare_keys},
})
df_all

## Section 7 — Key findings

In [ ]:
# ---------------------------------------------------------------------------
# Automated findings — derived programmatically from the runs above.
# ---------------------------------------------------------------------------
best_model_name = max(model_results, key=lambda n: model_results[n].metrics["sharpe_ratio"])
best_model_sharpe = float(model_results[best_model_name].metrics["sharpe_ratio"])

top3_features = mean_fi.head(3).index.tolist()

ml_sharpe  = float(r_ml.metrics["sharpe_ratio"])
mom_sharpe = float(r_mom.metrics["sharpe_ratio"])
mr_sharpe  = float(r_mr.metrics["sharpe_ratio"])

corr_ml_mom = float(corr_df.loc["ML", "Momentum"])
corr_ml_mr  = float(corr_df.loc["ML", "Mean Reversion"])
corr_mom_mr = float(corr_df.loc["Momentum", "Mean Reversion"])

low_corr_pairs = []
if corr_ml_mom < 0.3:  low_corr_pairs.append("ML x Momentum")
if corr_ml_mr  < 0.3:  low_corr_pairs.append("ML x Mean Reversion")
if corr_mom_mr < 0.3:  low_corr_pairs.append("Momentum x Mean Reversion")

ml_cost  = float(r_ml.metrics.get("total_cost_pct", 0))
mom_cost = float(r_mom.metrics.get("total_cost_pct", 0))
mr_cost  = float(r_mr.metrics.get("total_cost_pct", 0))

SEP = "=" * 60
print(SEP)
print("ML SIGNAL STRATEGY FINDINGS")
print(SEP)
print(f"  Walk-forward windows   : {len(strategy.training_dates_)}")
print(f"  Best model type        : {best_model_name}  (Sharpe = {best_model_sharpe:.4f})")
print(f"  Top 3 features         : {', '.join(top3_features)}")
print()
print(f"  Sharpe comparison:")
print(f"    ML (XGBoost)         : {ml_sharpe:.4f}")
print(f"    Momentum             : {mom_sharpe:.4f}")
print(f"    Mean Reversion       : {mr_sharpe:.4f}")
print()
print(f"  Pairwise return correlations:")
print(f"    ML <-> Momentum      : {corr_ml_mom:.4f}")
print(f"    ML <-> Mean Reversion: {corr_ml_mr:.4f}")
print(f"    Mom <-> Mean Rev     : {corr_mom_mr:.4f}")
if low_corr_pairs:
    print(f"  Low-correlation pairs (< 0.3): {', '.join(low_corr_pairs)}")
    print(f"  -> These pairs signal strong diversification for Phase 4.1")
else:
    print(f"  No pairs have correlation < 0.3")
    print(f"  -> Combining adds moderate diversification benefit")
print()
print(f"  Total cost drag:")
print(f"    ML                   : {ml_cost:.4f}%")
print(f"    Momentum             : {mom_cost:.4f}%")
print(f"    Mean Reversion       : {mr_cost:.4f}%")

## Notes

_Fill in manual observations after running the notebook:_

- Walk-forward windows and first prediction date:
- Best model type and margin over others:
- Most predictive features and intuition:
- Correlation with momentum/mean-reversion and Phase 4.1 implication:
- Cost drag relative to simpler strategies:
- Feature importance drift — did top features change over time?